# Comma → Verb

Cross-type attention: how much **Comma** tokens attend to **Verb** tokens across all 24 heads.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    get_type_positions, _values_entropy_normalized,
    show_cross_tokens, show_top_cross_pairs,
    CROSS_TYPE_NAMES,
)

In [ ]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)

## Matched Tokens

In [ ]:
show_cross_tokens(str_tokens, "comma", "verb")

## Comma → Verb — All 24 Heads

In [ ]:
type_positions = get_type_positions(str_tokens)
from_pos = type_positions["comma"]
to_pos = type_positions["verb"]
if not from_pos or not to_pos:
    print("No positions found for one or both types.")
else:
    results = []
    for layer in range(2):
        for head in range(12):
            a = get_attention_pattern(cache, layer, head)
            pct = a[from_pos][:, to_pos].sum(dim=-1).mean().item() * 100
            values = a[from_pos][:, to_pos].sum(dim=-1)
            ent = _values_entropy_normalized(values) * 100
            results.append(((layer, head), pct, ent))
    results.sort(key=lambda x: x[1], reverse=True)
    lines = ["| Head | Comma → Verb % | Entropy % |", "|------|-------|-------|"]  
    for (l, h), pct, ent in results:
        lines.append(f"| L{l}H{h} | {pct:.2f}% | {ent:.2f}% |")
    display(Markdown("\n".join(lines)))

## Top Heads

In [ ]:
if from_pos and to_pos:
    for (l, h), pct, ent in results[:3]:
        display(Markdown(f"---\n### L{l}H{h} — {pct:.2f}% (ent {ent:.2f}%)"))
        show_head_pattern(str_tokens, cache, layer=l, head=h)
        display(Markdown("#### Top Comma → Verb token pairs"))
        show_top_cross_pairs(str_tokens, cache, l, h, from_pos, to_pos, top_k=10)
        attention = get_attention_pattern(cache, layer=l, head=h)
        show_attention_tables(str_tokens, attention, top_k=15)